<a href="https://colab.research.google.com/github/hiba-ai-ing/projet-mL/blob/main/SABIR_HIBA_TP5_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Travaux Pratiques N°5 : Optimisation et comparaison des modèles CART et Random Forest en régression**

**Partie I : Questions de réflexion**


Q1) Complétez le tableau comparatif

| Critères de comparaison | cross_validate | GridSearchCV |
|---|---|---|
| Objectif principal | Évaluer et comparer des configurations définies manuellement (A, B, C), chacune par validation croisée à 5 plis | Trouver automatiquement la meilleure combinaison d'hyperparamètres parmi une grille de valeurs |
| Définition des hyperparamètres | Manuelle : l'utilisateur crée lui-même chaque modèle avec ses hyperparamètres (3 configurations A, B, C de complexité croissante) | Par une grille `param_grid` : dictionnaire de listes de valeurs pour plusieurs hyperparamètres (ex. 3 × 2 × 3 = 18 combinaisons) |
| Recherche automatique des meilleurs hyperparamètres | Non : seules les configurations choisies à la main sont testées, et c'est l'utilisateur qui compare les résultats | Oui : toutes les combinaisons de la grille sont testées automatiquement |
| Utilisation de la validation croisée | Oui : chaque configuration est évaluée sur les 5 plis (`cv=cv`, `scoring=metriques`, `return_train_score=True`) | Oui : chaque combinaison est évaluée sur les 5 mêmes plis (nombre d'évaluations = nombre de combinaisons × 5) |
| Entraînement des modèles (fit) | Pas de `fit` explicite : `cross_validate` entraîne 5 copies du modèle (une par pli) ; le modèle d'origine reste non entraîné | `recherche.fit(X_train, y_train)` entraîne chaque combinaison sur chaque pli, puis réentraîne le meilleur modèle sur tout `X_train` |
| Résultats et attributs accessibles | Dictionnaire de scores par pli : `test_MAE`, `test_RMSE`, `test_R2`, `train_R2`, etc. ; les moyennes et le tableau comparatif sont à construire à la main | `cv_results_`, `best_params_`, `best_score_`, `best_estimator_` ; `cv_results_` permet de construire le tableau de toutes les combinaisons |
| Sélection et réentraînement du modèle final | Sélection manuelle de la meilleure configuration (ex. B, selon le R² CV) ; le réentraînement doit être fait à la main | Sélection automatique selon le critère `refit="R2"` et réentraînement automatique de `best_estimator_`, prêt pour `predict()` |

**Q2) Expliquez le principe de fonctionnement de l'algorithme CART en régression, depuis la
construction de l'arbre jusqu'à la prédiction d'une nouvelle observation. Précisez comment les
variables et les seuils de division sont sélectionnés, le rôle du MSE dans ce processus et la manière
dont chaque feuille fournit une prédiction**


1. Idée générale

CART (Classification And Regression Trees) construit un arbre binaire par partitionnement récursif de l’espace des variables. On part d’un seul nœud contenant toutes les observations (la racine) et on le découpe progressivement en groupes de plus en plus homogènes, c’est-à-dire des groupes dans lesquels les valeurs de y sont aussi proches que possible.

2. Le MSE : mesure de dispersion d’un nœud

Pour un nœud A contenant n_A observations, on définit :

$$MSE(A) = \frac{1}{n_A}\sum_{i \in A} (y_i - \bar{y}_A)^2$$

où ȳ_A est la moyenne des y dans le nœud.

MSE faible : les y sont proches les uns des autres, donc le groupe est homogène.
MSE élevé : les y sont très dispersés, donc le nœud est hétérogène.

À la racine, le MSE est calculé sur les 28 observations. L’objectif de CART est de réduire cette erreur par des divisions successives.

3. Recherche de la meilleure division

Une division est définie par un couple (variable Xⱼ, seuil s). Elle envoie les observations vérifiant Xⱼ ≤ s dans le nœud gauche (G) et les autres dans le nœud droit (D).

Critère d’évaluation : le MSE pondéré par la taille des groupes

$$L(j,s) = \frac{n_G}{n_A}\,MSE(G) + \frac{n_D}{n_A}\,MSE(D)$$

CART retient le couple (j, s) qui minimise L(j,s).

Génération des seuils candidats pour chaque variable Xⱼ :

récupérer les valeurs de Xⱼ dans le nœud ;
supprimer les doublons et trier par ordre croissant ;
calculer les milieux entre deux valeurs consécutives : ce sont les seuils candidats ;
calculer L(j,s) pour chaque seuil et pour chaque variable ;
garder la variable et le seuil qui donnent la plus petite perte.

La sélection de la variable et du seuil est donc faite simultanément, par une recherche exhaustive (algorithme « glouton » : on optimise chaque division localement).

4. Critères d’arrêt

La construction continue dans les nœuds pouvant encore être divisés jusqu’à ce qu’un critère d’arrêt soit atteint, par exemple :

un nœud ne contient qu’une seule observation (ou moins qu’un minimum fixé) ;
le MSE du nœud est nul (ou très faible) ;
une profondeur maximale est atteinte ;
la baisse de L obtenue par une division est insuffisante.

Sans limitation, l’arbre peut aller jusqu’à une observation par feuille, ce qui conduit au surapprentissage. On contrôle cela par ces critères (pré-élagage) ou par un élagage a posteriori.

5. Prédiction d’une feuille

Chaque feuille F fournit une prédiction constante : la moyenne des y des observations qu’elle contient

$$\hat{y}_F = \bar{y}_F = \frac{1}{n_F}\sum_{i \in F} y_i$$

Cette valeur est celle qui minimise le MSE dans la feuille. Par exemple, dans l’exemple du cours, toute observation tombant dans G1 (avant sa division) aurait été prédite à 100,11, et dans D1 à 951,63. Une feuille à une seule observation prédit directement la valeur observée.

6. Prédire une nouvelle observation

Pour une nouvelle observation (x₁, x₂) :

on part de la racine ;
on teste la condition du nœud (ex. X₂ ≤ 198,75) : si oui, on va à gauche, sinon à droite ;
on répète le test à chaque nœud interne, en descendant dans l’arbre ;
on s’arrête dans une feuille ;
la prédiction est la moyenne ȳ de cette feuille.

Exemple avec l’arbre de l’étape 6 : pour une observation avec X₂ = 250 et X₁ = 40, le test X₂ ≤ 198,75 est faux, donc on va dans D1. Le test X₁ ≤ 22,04 est faux, donc on va dans D2. Le test X₁ ≤ 52,28 est vrai, donc on arrive dans G6, et la prédiction est la moyenne des y des 6 observations de G6.

**Q3) Expliquez le principe du Bootstrap (tirage avec remise) utilisé dans Random Forest.
Comment ce mécanisme contribue-t-il à construire des arbres différents ? Expliquez ensuite
pourquoi la combinaison des prédictions de plusieurs arbres permet d'atténuer l'effet escalier
observé avec un arbre CART unique, sans nécessairement le supprimer complètement**

1. Principe du Bootstrap

Pour chaque arbre, on construit un échantillon d’apprentissage en tirant N observations avec remise parmi les N du jeu d’entraînement. Certaines observations apparaissent plusieurs fois, d’autres ne sont pas tirées : ce sont les observations OOB (Out-Of-Bag). Elles servent à évaluer l’arbre sur des données qu’il n’a pas vues, et leur combinaison donne une estimation interne de la généralisation (oob_score=True, avec bootstrap=True).

2. Arbres différents

Un arbre CART est instable : une petite modification des données change les seuils et la structure. Chaque arbre étant entraîné sur un échantillon différent, les MSE pondérés L(j,s) changent, donc le meilleur couple (variable, seuil) n’est pas le même d’un arbre à l’autre. Le sous-échantillonnage des variables (max_features) renforce cette diversité. Les arbres sont ainsi différents et moins corrélés.

3. Atténuation de l’effet escalier

Un arbre unique donne une prédiction constante par morceaux (moyenne de chaque feuille), avec des sauts brusques aux seuils. En régression, la forêt moyenne les prédictions des arbres. Comme chaque arbre a ses propres seuils, les marches sont décalées : un grand saut devient plusieurs petits sauts. La courbe est plus lisse et plus stable, et la variance diminue.

4. Pas de suppression complète
La moyenne de fonctions en escalier reste une fonction en escalier (marches plus fines).
Les arbres restent partiellement corrélés (même jeu de données d’origine) : certaines coupures dominantes se répètent.
Le nombre d’arbres est fini.

q4)

| Hyperparamètre | Rôle, valeur par défaut et influence sur le modèle |
|---|---|
| n_estimators | **Rôle :** nombre d'arbres construits dans la forêt. **Défaut :** 100. **Influence :** plus il est grand, plus la moyenne des arbres est stable (variance réduite), au prix d'un temps de calcul plus élevé. |
| max_depth | **Rôle :** profondeur maximale de chaque arbre. **Défaut :** None (aucune limite). **Influence :** une profondeur élevée rend les arbres plus flexibles (complexité ↑) mais augmente leur variance et le risque de surapprentissage. |
| min_samples_split | **Rôle :** nombre minimal d'observations dans un nœud pour autoriser sa division. **Défaut :** 2. **Influence :** une valeur plus élevée limite la croissance des arbres (complexité ↓) et favorise la généralisation. |
| min_samples_leaf | **Rôle :** nombre minimal d'observations autorisé dans chaque feuille. **Défaut :** 1. **Influence :** une valeur plus élevée évite les feuilles contenant très peu d'observations et rend les prédictions plus stables (complexité ↓, généralisation ↑). |
| max_features | **Rôle :** nombre de variables considérées lors de la recherche du meilleur split à chaque nœud. **Défaut :** 1.0 (toutes les variables). **Influence :** une valeur plus faible augmente la diversité entre les arbres et réduit leur corrélation. |
| bootstrap | **Rôle :** indique si chaque arbre est entraîné sur un échantillon obtenu par tirage avec remise. **Défaut :** True. **Influence :** si True, chaque arbre voit un échantillon différent (diversité ↑) ; si False, chaque arbre utilise toutes les observations. |
| oob_score | **Rôle :** active l'évaluation Out-Of-Bag : les observations non tirées par le bootstrap servent à estimer la performance de généralisation. **Défaut :** False. **Influence :** n'affecte ni la complexité ni la diversité ; nécessite bootstrap=True. |
| criterion | **Rôle :** critère utilisé pour évaluer la qualité des coupures. **Défaut :** "squared_error" (MSE). **Influence :** "absolute_error" repose sur l'erreur absolue et est généralement moins sensible aux valeurs extrêmes. |

.











**Partie II : Étude de cas — Prédiction de la valeur des logements**

**A. Préparation et exploration des données**

Q1) Importez les bibliothèques nécessaires à la manipulation et à la visualisation des données, à
l'implémentation des modèles CART et Random Forest, à l'optimisation des hyperparamètres avec
GridSearchCV et à l'évaluation des performances. Fixez SEED=42 pour assurer la
reproductibilité.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor

from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
)

# Reproductibilité
SEED = 42
np.random.seed(SEED)



**Q2) Chargez le jeu de données California Housing à partir du fichier CSV disponible au lien suivant
https://codesignal-learn.github.io/course_building-aws-sagemaker/california_housing.csv**

In [17]:
#dataset importation
dataset=pd.read_csv('california_housing.csv' )

# Sélection aléatoire de 8 000 observations
dataset2 = dataset.sample(n=8000, random_state=SEED)

# Dimensions
print('----le nombres de lignes---- ')
print(dataset2.shape[0])
print('----le nombre de colonnes---------')
print(dataset2.shape[1])

# Cinq premières lignes
print('--------les cinq premiers lignes:-------',dataset2.head())

# Types des variables
print('-------les types des variables:-------------',dataset2.dtypes)

# Statistiques descriptives
print('----------statistique descriptives-----------',dataset2.describe())

# Valeurs manquantes
print( '-------le nombres de valeur manguantes------------',dataset2.isna().sum())

----le nombres de lignes---- 
8000
----le nombre de colonnes---------
9
--------les cinq premiers lignes:-------        MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
20046  1.6812      25.0  4.192201   1.022284      1392.0  3.877437     36.06   
3024   2.5313      30.0  5.039384   1.193493      1565.0  2.679795     35.14   
15663  3.4801      52.0  3.977155   1.185877      1310.0  1.360332     37.80   
20484  5.7376      17.0  6.163636   1.020202      1705.0  3.444444     34.28   
9814   3.7250      34.0  5.492991   1.028037      1063.0  2.483645     36.62   

       Longitude  MedHouseVal  
20046    -119.01      0.47700  
3024     -119.46      0.45800  
15663    -122.44      5.00001  
20484    -118.72      2.18600  
9814     -121.93      2.78000  
-------les types des variables:------------- MedInc         float64
HouseAge       float64
AveRooms       float64
AveBedrms      float64
Population     float64
AveOccup       float64
Latitude       float64
Longitud

**Interprétation des résultats**
1. Dimensions

Le jeu contient 8 000 lignes et 9 colonnes : 8 variables explicatives (MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude) et 1 cible (MedHouseVal). C’est la version scikit-learn de California Housing, donc le nom MedHouseVal utilisé en Q3 est correct. Après le split 85/15, on aura 6 800 observations d’entraînement et 1 200 de test.

2. Cinq premières lignes

Les index (20046, 3024, 15663...) ne sont pas ordonnés, ce qui confirme que l’échantillonnage est aléatoire (pas de reset_index). On remarque aussi :

les variables sont à des échelles très différentes (revenu ~1 à 6, latitude ~35, longitude ~-120) ;
la ligne 15663 a MedHouseVal = 5.00001, valeur qui correspond au plafond de la cible.
3. Types des variables

Toutes les colonnes sont float64 : aucune variable catégorielle, donc aucun encodage n’est nécessaire. Les modèles CART et Random Forest peuvent être entraînés directement.

4. Statistiques descriptives
MedInc : moyenne 3,86, médiane 3,54, max 15,0. La moyenne est supérieure à la médiane et le max est très éloigné du 75e centile (4,73) : distribution asymétrique à droite, avec quelques revenus très élevés.
HouseAge : moyenne 28,8, médiane 29, plage de 1 à 52 ans. Distribution assez symétrique, avec un maximum de 52 (valeur plafonnée).
Longitude : de -124,3 à -114,58, avec un 25e centile à -121,76 et une médiane à -118,47. Les données se concentrent autour de deux zones (Bay Area et Los Angeles), ce qui explique le saut entre quartiles.
MedHouseVal (cible) : moyenne 2,07 (≈ 207 000 $), médiane 1,81, écart-type 1,16, min 0,15, max 5,00001. Moyenne > médiane : distribution asymétrique à droite. Le maximum à 5,00001 signale un plafonnement (censure) : toutes les maisons de valeur supérieure à 500 000 $ sont enregistrées à cette valeur.

Les colonnes AveRooms, AveBedrms, Population, AveOccup sont tronquées à l’affichage (...). Affichez-les avec dataset.describe().T pour voir leurs maxima : elles contiennent en général des valeurs extrêmes (ex. AveOccup de plusieurs centaines).

5. Valeurs manquantes

Aucune valeur manquante (0 pour toutes les colonnes) : ni imputation ni suppression de lignes n’est nécessaire.

**Q3) Séparez les variables explicatives X et la cible y, puis partitionnez les données en 85 % pour
l'entraînement et 15 % pour le test, avec random_state=SEED. Vérifiez les dimensions des
ensembles obtenus**

In [18]:
# Variables explicatives (X) et cible (y)
X = dataset2.drop(columns=["MedHouseVal"])
y = dataset2["MedHouseVal"]

# Partition 85 % entraînement / 15 % test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED
)

# Vérification des dimensions
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (6800, 8)
X_test  : (1200, 8)
y_train : (6800,)
y_test  : (1200,)


X_train : (6800, 8) et y_train : (6800,) : 85 % de 8 000 = 6 800 observations d’entraînement, avec 8 variables explicatives.
X_test : (1200, 8) et y_test : (1200,) : 15 % de 8 000 = 1 200 observations de test.
6 800 + 1 200 = 8 000, donc aucune observation n’est perdue ou dupliquée.
X et y ont le même nombre de lignes dans chaque ensemble, donc chaque observation garde sa cible.

**Q4) Définissez une validation croisée à cinq plis avec KFold et mélange des observations. Préparez
un dictionnaire contenant les trois métriques MAE, RMSE et R². Expliquez l'utilisation des scores
négatifs pour MAE et RMSE.**

In [19]:
# Validation croisée à 5 plis avec mélange des observations
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

# Dictionnaire des trois métriques (MAE, RMSE, R²)
metriques = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

Dans scikit-learn, tous les scorers suivent la convention « plus grand = meilleur ». cross_validate et GridSearchCV cherchent donc toujours à maximiser le score (refit="R2", best_score_, etc.).

R² : plus il est élevé, meilleur est le modèle, donc il est utilisé tel quel.
MAE et RMSE sont des erreurs : plus elles sont faibles, meilleur est le modèle. Pour respecter la convention, scikit-learn renvoie leur opposé (neg_), c’est-à-dire −MAE et −RMSE. Le meilleur modèle est alors celui dont le score est le plus proche de 0, donc le plus grand.

pour avoir de valeurs négatifs , on peut aussi utilisé la methode du cours , mais cette methode reste  plus optimale.

**B. Optimisation du modèle CART**


**Q5) Construisez un modèle DecisionTreeRegressor et définissez la grille
d'hyperparamètres suivante :**





In [21]:
# Modèle de base
modele1 = DecisionTreeRegressor(random_state=SEED)

# Grille : 3 x 2 x 3 = 18 combinaisons
param_grid1 = {
    "max_depth": [6, 12, None],
    "min_samples_split": [2, 30],
    "min_samples_leaf": [1, 5, 10]
}

# GridSearchCV : 5 plis, 3 métriques, R² comme critère principal
recherche1 = GridSearchCV(
    estimator=modele1,
    param_grid=param_grid1,
    scoring=metriques,
    refit="R2",
    cv=cv,
    n_jobs=1
)

recherche1.fit(X_train, y_train)

# Résultats
print("Nombre de combinaisons :", len(recherche1.cv_results_["params"]))
print("Nombre d'evaluations CV :", len(recherche1.cv_results_["params"]) * 5)
print("Meilleurs parametres :", recherche1.best_params_)
print("Meilleur R2 moyen CV :", round(recherche1.best_score_, 4))

Nombre de combinaisons : 18
Nombre d'evaluations CV : 90
Meilleurs parametres : {'max_depth': 12, 'min_samples_leaf': 10, 'min_samples_split': 30}
Meilleur R2 moyen CV : 0.6794


**Interprétation des résultats de GridSearchCV (CART)**
1. Combinaisons et évaluations
18 combinaisons : 3 (max_depth) × 2 (min_samples_split) × 3 (min_samples_leaf).
90 évaluations : 18 × 5 plis. Chaque combinaison est entraînée et validée 5 fois. Ensuite, best_estimator_ est réentraîné une fois sur tout X_train (refit="R2").
2. Meilleurs hyperparamètres

max_depth = 12, min_samples_split = 30, min_samples_leaf = 10.

min_samples_split = 30 et min_samples_leaf = 10 : ce sont les valeurs les plus contraignantes de la grille. Elles interdisent les feuilles de moins de 10 observations et les divisions de nœuds de moins de 30 observations. Les feuilles prédisent donc une moyenne calculée sur plusieurs observations, ce qui rend l’arbre moins sensible au bruit.
max_depth = 12 : l’arbre garde une profondeur relativement élevée. Comme min_samples_leaf et min_samples_split limitent déjà la croissance, la profondeur 12 est rarement atteinte en pratique. Elle ne pénalise donc pas le modèle.
Les valeurs retenues sont les plus régularisantes de la grille pour min_samples_split et min_samples_leaf. Le meilleur modèle n’est donc pas le plus flexible : cela traduit un surapprentissage des arbres profonds (feuilles à 1 observation) qui mémorisent le bruit.
3. Meilleur R² moyen CV = 0,6794

Le modèle explique environ 68 % de la variance de la valeur des maisons sur des données non utilisées pour l’entraînement de chaque pli. C’est une performance correcte mais limitée pour un arbre unique :

environ 32 % de la variance reste inexpliquée ;
l’arbre est instable et produit une prédiction en escalier (constante par feuille) ;
le plafonnement de la cible à 5,0 et la présence de valeurs extrêmes pénalisent aussi le MSE.
4. Remarque importante

Une valeur au bord de la grille (30 pour min_samples_split, 10 pour min_samples_leaf) suggère que la régularisation optimale pourrait être encore plus forte. On pourrait élargir la grille (par exemple min_samples_leaf : 10, 20, 30, min_samples_split : 30, 50) pour vérifier. Pour cet exercice, la grille imposée suffit.

**Q6) Construisez un tableau présentant les 18 configurations de CART, leurs hyperparamètres ainsi
que les moyennes des scores MAE, RMSE et R² en validation croisée et le R² moyen sur les plis
d'entraînement. Classez les configurations par R² de validation décroissant. Analysez l'influence
des hyperparamètres et les éventuels signes de surapprentissage**

In [23]:
# Relancer GridSearchCV avec les scores d'entraînement
recherche1 = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=SEED),
    param_grid=param_grid1,
    scoring=metriques,
    refit="R2",
    cv=cv,
    #voila l'hyperparametre ajouté:
    return_train_score=True,
    n_jobs=1
)
recherche1.fit(X_train, y_train)

# Tableau des 18 configurations (même méthode que le cours)
resultats_cart = pd.DataFrame(recherche1.cv_results_)
resultats_cart["MAE_CV"] = -resultats_cart["mean_test_MAE"]
resultats_cart["RMSE_CV"] = -resultats_cart["mean_test_RMSE"]
resultats_cart["R2_CV"] = resultats_cart["mean_test_R2"]
resultats_cart["R2_Train_CV"] = resultats_cart["mean_train_R2"]

colonnes = [
    "param_max_depth", "param_min_samples_split", "param_min_samples_leaf",
    "MAE_CV", "RMSE_CV", "R2_CV", "R2_Train_CV"]

print(resultats_cart[colonnes]
      .sort_values("R2_CV", ascending=False)
      .round(3).to_string(index=False))

param_max_depth  param_min_samples_split  param_min_samples_leaf  MAE_CV  RMSE_CV  R2_CV  R2_Train_CV
             12                       30                      10   0.448    0.654  0.679        0.808
           None                       30                      10   0.448    0.654  0.679        0.814
             12                        2                      10   0.449    0.656  0.677        0.824
           None                        2                      10   0.448    0.656  0.677        0.832
             12                       30                       5   0.450    0.660  0.673        0.820
           None                       30                       5   0.455    0.666  0.668        0.830
             12                       30                       1   0.455    0.677  0.656        0.837
           None                       30                       1   0.461    0.686  0.647        0.856
             12                        2                       5   0.463    0.688 

Pour obtenir le R² sur les plis d’entraînement, GridSearchCV doit être lancé avec return_train_score=True. Ce n’était pas le cas en Q5, il faut donc relancer la recherche (même résultat, avec les scores d’entraînement en plus).

Lecture générale

Le classement va de R² CV = 0,679 (max_depth=12, min_samples_split=30, min_samples_leaf=10) à 0,543 (max_depth=None, min_samples_split=2, min_samples_leaf=1). La meilleure configuration a un MAE de 0,448 et un RMSE de 0,654 (en centaines de milliers de dollars, soit environ 44 800 $ et 65 400 $ d’erreur).

Influence des hyperparamètres

1. min_samples_leaf (le plus influent à profondeur élevée)
Pour max_depth = 12 et min_samples_split = 30 : leaf = 1 → R² CV 0,656, leaf = 5 → 0,673, leaf = 10 → 0,679. Le R² de validation augmente avec la taille minimale des feuilles, alors que le R² d’entraînement baisse (0,837 → 0,820 → 0,808). Des feuilles qui moyennent plus d’observations réduisent la variance.

2. min_samples_split
Son effet est faible quand leaf = 10 (0,679 contre 0,677 pour 30 contre 2). Il compte davantage quand leaf est petit : avec max_depth=12 et leaf=5, passer de 2 à 30 fait monter R² CV de 0,645 à 0,673. Avec leaf = 10, un nœud doit déjà contenir 20 observations pour être divisé, donc la contrainte split = 30 est en partie redondante.

3. max_depth

Profondeur 6 : R² train ≈ R² CV (0,694 contre 0,633) et les résultats varient très peu selon leaf et split (0,621 à 0,633). La profondeur limite alors l’arbre avant les autres contraintes : c’est du sous-apprentissage (biais élevé).
Profondeur 12 et None : donnent des résultats très proches quand leaf est grand (0,679 contre 0,679), car leaf et split arrêtent la croissance avant la profondeur maximale. La différence apparaît avec les contraintes faibles : à leaf = 1 et split = 2, on passe de 0,599 (profondeur 12) à 0,543 (None).


Surapprentissage net pour l’arbre non contraint (None, 2, 1) : R² train = 1,000 signifie que chaque observation d’entraînement a sa propre feuille, l’arbre mémorise les données, y compris le bruit. Il est pourtant le pire en validation.
Plus la contrainte se relâche, plus l’écart train/validation augmente, et le R² CV diminue.
Les arbres de profondeur 6 ont l’écart le plus faible (environ 0,06 à 0,08), mais leur R² CV reste plus bas : trop simples pour capter la structure des données.
La meilleure configuration se trouve au compromis biais-variance : assez flexible (profondeur 12), mais régularisée par les feuilles de taille ≥ 10. Son écart de 0,13 montre qu’un surapprentissage modéré subsiste.

**C. Optimisation du modèle Random Forest**


**Q7) Construisez un modèle RandomForestRegressor comprenant 100 arbres, avec
bootstrap=True et max_features=0.8**

In [13]:
# Modèle de base : 100 arbres, bootstrap, 80 % des variables à chaque split
modele2 = RandomForestRegressor(
    n_estimators=100,
    bootstrap=True,
    max_features=0.8,
    random_state=SEED
)

# Grille : 3 x 2 x 3 = 18 combinaisons
param_grid2 = {
    "max_depth": [8, 16, None],
    "min_samples_split": [2, 30],
    "min_samples_leaf": [1, 5, 10]
}

# GridSearchCV : mêmes plis (cv), mêmes métriques, R² comme critère principal
recherche2 = GridSearchCV(
    estimator=modele2,
    param_grid=param_grid2,
    scoring=metriques,
    refit="R2",
    cv=cv,
    return_train_score=True,
    n_jobs=1
)

recherche2.fit(X_train, y_train)

# Résultats
print("Nombre de combinaisons :", len(recherche2.cv_results_["params"]))
print("Nombre d'evaluations CV :", len(recherche2.cv_results_["params"]) * 5)
print("Meilleurs parametres :", recherche2.best_params_)
print("Meilleur R2 moyen CV :", round(recherche2.best_score_, 4))

Nombre de combinaisons : 18
Nombre d'evaluations CV : 90
Meilleurs parametres : {'max_depth': 16, 'min_samples_leaf': 1, 'min_samples_split': 2}
Meilleur R2 moyen CV : 0.7796


2. Meilleurs hyperparamètres

max_depth = 16, min_samples_leaf = 1, min_samples_split = 2.

C’est la configuration la moins contrainte de la grille (feuilles à 1 observation, division dès 2 observations), alors que pour CART la meilleure était la plus régularisée (leaf = 10, split = 30). La différence s’explique par le moyennage : chaque arbre profond a une variance élevée, mais les erreurs de 100 arbres décorrélés (bootstrap + max_features=0.8) se compensent. La forêt n’a donc pas besoin de limiter la croissance de chaque arbre.

max_depth = 16 est retenu plutôt que None, probablement avec un score presque identique : à partir d’une certaine profondeur, les arbres sont de toute façon arrêtés par le manque d’observations.

3. Meilleur R² moyen CV = 0,7796

Le modèle explique environ 78 % de la variance de la valeur des maisons sur des données non vues par chaque pli.

**Q8) Présentez les performances des 18 configurations de Random Forest**

In [24]:
# Tableau des 18 configurations Random Forest
res2 = pd.DataFrame(recherche2.cv_results_)
res2["MAE_CV"] = -res2["mean_test_MAE"]
res2["RMSE_CV"] = -res2["mean_test_RMSE"]
res2["R2_CV"] = res2["mean_test_R2"]
res2["R2_Train_CV"] = res2["mean_train_R2"]
res2["Ecart_R2"] = res2["R2_Train_CV"] - res2["R2_CV"]

colonnes = [
    "param_max_depth", "param_min_samples_split", "param_min_samples_leaf",
    "MAE_CV", "RMSE_CV", "R2_CV", "R2_Train_CV", "Ecart_R2"]

print(res2[colonnes]
      .sort_values("R2_CV", ascending=False)
      .round(3).to_string(index=False))

param_max_depth  param_min_samples_split  param_min_samples_leaf  MAE_CV  RMSE_CV  R2_CV  R2_Train_CV  Ecart_R2
             16                        2                       1   0.366    0.542  0.780        0.964     0.185
           None                        2                       1   0.366    0.545  0.778        0.969     0.191
             16                        2                       5   0.371    0.549  0.774        0.893     0.119
           None                        2                       5   0.371    0.549  0.774        0.893     0.119
             16                       30                       1   0.383    0.561  0.764        0.863     0.099
           None                       30                       1   0.384    0.562  0.764        0.864     0.100
           None                        2                      10   0.384    0.563  0.762        0.843     0.081
             16                       30                       5   0.385    0.563  0.762        0.843   

Meilleure configuration :

max_depth = 16, min_samples_split = 2, min_samples_leaf = 1,

 avec R² CV = 0,780, MAE = 0,366 et RMSE = 0,542.

Le modèle explique environ 78 % de la variance de MedHouseVal, avec une erreur moyenne d’environ 36 600 $.

Sensibilité aux hyperparamètres : le R² CV varie de seulement 0,739 à 0,780 sur les 18 configurations, donc la forêt est peu sensible au réglage.

max_depth : 16 et None sont équivalents. 8 est en retrait (arbres trop simples).
min_samples_leaf : plus il augmente, plus le R² CV baisse légèrement (0,780 → 0,762).
min_samples_split : effet faible.

Surapprentissage : l’écart train/validation atteint 0,185 pour la meilleure configuration (R² train = 0,964). Il s’agit d’un surapprentissage apparent, sans effet sur la validation : la régularisation réduit l’écart mais ne améliore pas le R² CV. Le bootstrap, max_features = 0.8 et le moyennage contrôlent déjà la variance.

Limites : environ 22 % de variance reste inexpliquée (plafonnement de la cible, valeurs extrêmes). La performance doit encore être confirmée sur le jeu de test.

**D. Comparaison des modèles et évaluation finale**


**Q9) Comparez les meilleures configurations de CART et Random Forest**

| Modèle | MAE_CV | RMSE_CV | R² CV | R² Train | Écart (train − CV) |
|---|---|---|---|---|---|
| CART (12, 30, 10) | 0,448 | 0,654 | 0,679 | 0,808 | 0,129 |
| Random Forest (16, 2, 1) | 0,366 | 0,542 | 0,780 | 0,964 | 0,185 |

Comparaison des performances en validation croisée

MAE. La forêt obtient une erreur absolue moyenne de 0,366 contre 0,448 pour CART, soit environ 18 % de moins. Comme la cible est exprimée en centaines de milliers de dollars, cela correspond à une erreur moyenne d’environ 36 600 $ contre 44 800 $.

RMSE. La forêt obtient 0,542 contre 0,654 (environ 17 % de moins). Le RMSE pénalise davantage les grosses erreurs, donc la forêt se trompe moins souvent de façon importante, en particulier sur les maisons chères ou au plafond de 5,0.

R². La forêt atteint 0,780 contre 0,679, soit un gain de 0,101 point. Elle explique environ 78 % de la variance de la valeur des maisons sur des données non vues, contre 68 % pour CART. La variance non expliquée passe ainsi d’environ 32 % à 22 %.

Robustesse de la conclusion. La forêt est meilleure sur les trois métriques. De plus, même la moins bonne forêt de la grille (8, 30, 1 : R² CV = 0,739) dépasse le meilleur CART (0,679). L’avantage ne dépend donc pas d’un réglage chanceux.

3. Performances d’entraînement et risque de surapprentissage

CART. Le R² train est de 0,808 pour un R² CV de 0,679, soit un écart de 0,129. Ce surapprentissage est modéré, car la meilleure configuration est fortement régularisée (feuilles d’au moins 10 observations). Sans régularisation (None, 2, 1), l’arbre atteint un R² train de 1,000 mais un R² CV de seulement 0,543 : il mémorise les données d’entraînement, y compris le bruit, et généralise mal. Pour CART, le surapprentissage dégrade donc directement la validation, et c’est pourquoi la meilleure configuration est la plus contrainte.

Random Forest. Le R² train est de 0,964 pour un R² CV de 0,780, soit un écart de 0,185, plus élevé que celui de CART. Le surapprentissage apparent est donc plus important. Les arbres sont profonds (feuilles de 1 observation) et ajustent presque parfaitement l’entraînement. Mais, contrairement à CART, cet écart ne dégrade pas la validation : la configuration qui a le plus grand écart est aussi celle qui a le meilleur R² CV. Le bootstrap, max_features = 0.8 et le moyennage de 100 arbres décorrélés compensent la forte variance de chaque arbre.

Lecture correcte de l’écart. Le R² train est optimiste, surtout pour une forêt dont les arbres ont vu les observations d’entraînement. L’écart train/validation indique un risque, mais il ne suffit pas à choisir un modèle. Le critère décisif est le R² de validation, qui mesure la capacité à généraliser.

4. Modèle retenu : Random Forest (16, 2, 1)

Justification :

Meilleur R² moyen de validation (0,780 contre 0,679), qui est le critère de sélection imposé.
Erreurs plus faibles : MAE et RMSE sont inférieurs sur les 5 plis en moyenne.
Plus grande stabilité : le R² CV de la forêt varie de 0,739 à 0,780 sur les 18 configurations, contre 0,543 à 0,679 pour CART. Le choix des hyperparamètres est donc moins risqué.
Variance réduite : le moyennage d’arbres décorrélés corrige l’instabilité d’un arbre unique et atténue l’effet escalier de ses prédictions.
5. Limites
Le R² train de 0,964 est optimiste et ne doit pas être présenté comme la performance réelle du modèle.
La forêt est moins interprétable qu’un arbre unique, que l’on peut lire règle par règle, et plus coûteuse en temps de calcul (100 arbres).
Environ 22 % de la variance reste inexpliquée, à cause du plafonnement de la cible à 5,0, des valeurs extrêmes et de variables absentes (qualité du quartier, etc.).
La performance doit être confirmée sur le jeu de test X_test (1 200 observations), jamais utilisé pendant la validation croisée, avec best_estimator_.

**Q10) Évaluez les deux modèles optimisés sur le jeu de test indépendant,**

In [25]:
# Modèles optimisés (déjà réentraînés sur X_train)
best_cart = recherche1.best_estimator_
best_rf = recherche2.best_estimator_

def evaluer(modele, X, y):
    pred = modele.predict(X)
    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": root_mean_squared_error(y, pred),
        "R2": r2_score(y, pred),
    }

# Scores sur le jeu de test
test_cart = evaluer(best_cart, X_test, y_test)
test_rf = evaluer(best_rf, X_test, y_test)

# Tableau comparatif : validation croisée vs test
comparaison = pd.DataFrame({
    "Modèle": ["CART", "Random Forest"],
    "MAE_CV": [-recherche1.cv_results_["mean_test_MAE"][recherche1.best_index_],
               -recherche2.cv_results_["mean_test_MAE"][recherche2.best_index_]],
    "MAE_Test": [test_cart["MAE"], test_rf["MAE"]],
    "RMSE_CV": [-recherche1.cv_results_["mean_test_RMSE"][recherche1.best_index_],
                -recherche2.cv_results_["mean_test_RMSE"][recherche2.best_index_]],
    "RMSE_Test": [test_cart["RMSE"], test_rf["RMSE"]],
    "R2_CV": [recherche1.best_score_, recherche2.best_score_],
    "R2_Test": [test_cart["R2"], test_rf["R2"]],
})
comparaison["Ecart_R2 (CV - Test)"] = comparaison["R2_CV"] - comparaison["R2_Test"]
print(comparaison.round(3).to_string(index=False))

       Modèle  MAE_CV  MAE_Test  RMSE_CV  RMSE_Test  R2_CV  R2_Test  Ecart_R2 (CV - Test)
         CART   0.448     0.448    0.654       0.65  0.679    0.682                -0.003
Random Forest   0.366     0.361    0.542       0.56  0.780    0.764                 0.015


Tableau de résultats
| Modèle | MAE_CV | MAE_Test | RMSE_CV | RMSE_Test | R² CV | R² Test | Écart R² (CV − Test) |
|---|---|---|---|---|---|---|---|
| CART | 0,448 | 0,448 | 0,654 | 0,650 | 0,679 | 0,682 | -0,003 |
| Random Forest | 0,366 | 0,361 | 0,542 | 0,560 | 0,780 | 0,764 | 0,015 |
2. Comparaison validation croisée / test

CART. Les scores test sont quasiment identiques à ceux de la validation croisée : MAE 0,448 contre 0,448, RMSE 0,650 contre 0,654, R² 0,682 contre 0,679. L’écart de R² (-0,003) est négligeable, et le test est même très légèrement meilleur. Ce n’est pas un vrai gain : avec 1 200 observations, le score test varie naturellement de quelques millièmes.

Random Forest. Les scores restent proches : MAE 0,361 (contre 0,366), RMSE 0,560 (contre 0,542), R² 0,764 (contre 0,780). Le R² baisse de 0,015 et le RMSE monte de 0,018, tandis que le MAE s’améliore légèrement. Cette combinaison suggère que le jeu de test contient quelques grosses erreurs (maisons chères ou au plafond de 5,0) que le RMSE pénalise davantage que le MAE.


**Capacité de généralisation**

Les deux modèles généralisent bien : l’écart de R² entre validation croisée et test est inférieur à 0,02, donc la validation croisée était une estimation fiable.
Le surapprentissage apparent de la forêt (écart train/validation de 0,185) est sans conséquence : son R² test (0,764) reste proche du R² CV (0,780), grâce au moyennage de 100 arbres décorrélés.
Le CART, fortement régularisé (feuilles d’au moins 10 observations), est très stable : le test reflète fidèlement la validation.
La légère baisse de la forêt (0,015) est cohérente avec la sélection du meilleur réglage parmi 18 configurations et reste dans la variabilité d’un test de 1 200 observations.

**Comparaison sur le test**

La forêt reste meilleure sur les trois métriques :

MAE : 0,361 contre 0,448 (≈ 19 % de moins) ;
RMSE : 0,560 contre 0,650 (≈ 14 % de moins) ;
R² : 0,764 contre 0,682 (+0,082).

L’avantage est un peu plus faible qu’en validation croisée (+0,101), mais reste net.

**Choix du modèle**

Le Random Forest (max_depth = 16, min_samples_split = 2, min_samples_leaf = 1), sélectionné en Q9 sur le R² CV, est confirmé par le test. Il n’est pas modifié : le test sert uniquement à mesurer la performance finale, et changer de modèle à partir de ses résultats biaiserait l’estimation.

**Q11) À partir des scores R² obtenus sur les ensembles d'entraînement, de validation croisée et de
test, analysez la capacité de généralisation du modèle retenu. Identifiez les éventuels signes de
surapprentissage et expliquez pourquoi GridSearchCV ne garantit pas leur élimination. Proposez
deux pistes d'amélioration pour réduire le surapprentissage tout en préservant les performances en
validation.**





| Ensemble | R² |
|---|---|
| Entraînement (moyenne des plis CV) | 0,964 |
| Validation croisée | 0,780 |
| Test | 0,764 |

Surapprentissage : l’écart train/validation (≈ 0,18) est le signe principal. Il vient d’arbres très profonds (max_depth = 16, feuilles de 1 observation) qui mémorisent en partie le bruit. Il reste modéré en pratique, car le moyennage de 100 arbres protège la validation et le test.

Pourquoi GridSearchCV ne l’élimine pas :

Il maximise le R² de validation, sans pénaliser l’écart train/validation.
La grille est limitée (18 combinaisons) et les valeurs sont imposées.
Choisir le meilleur parmi 18 configurations rend le score CV légèrement optimiste.
Le surapprentissage est une propriété du modèle : GridSearchCV compare des modèles, il ne les régularise pas.

Deux pistes d’amélioration :

Régulariser plus finement les arbres : min_samples_leaf entre 2 et 4, max_depth entre 10 et 14. Passer de leaf = 1 à 5 réduisait l’écart de 0,185 à 0,119 pour seulement 0,006 de R² CV perdu.
Augmenter la diversité des arbres : max_features plus faible (0,5 à 0,6), max_samples autour de 0,8, et éventuellement plus d’arbres (n_estimators = 300), pour mieux décorréler les arbres et réduire la variance.

et  toute modification se valide par validation croisée sur X_train, jamais à partir du test, qui reste un contrôle final non biaisé.



